# IndiaTaxBench — env-connected training (Qwen2.5 + OpenEnv Space)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prateedk/IndiaTaxBench-OpenEnv/blob/main/notebooks/train_qwen_india_tax.ipynb)

> **Colab tip:** the first code cell auto-clones this repo and installs `[notebook]` extras. Set Colab **Secrets** for `HF_TOKEN` (required to download Qwen weights), and optionally `ENV_URL`, `OPENAI_API_KEY`, `MODEL_NAME`. **Runtime → Change runtime type → GPU** (T4 is fine for 3B).

This notebook **downloads the base model locally**, collects **rollouts against your OpenEnv Hugging Face Space** (`ENV_URL`: `reset` / `step` → observations and **rewards**), builds **SFT-style** data from those trajectories, and runs a small **TRL `SFTTrainer`** pass (hackathon-style stub).

**Roles**

1. **`ENV_URL`** — Your IndiaTaxBench **Space** (FastAPI): environment transitions only. It does **not** run your training backward pass.
2. **Hub + local GPU** — `from_pretrained(MODEL_NAME)` and LoRA updates happen **on this machine** (Colab, HF GPU session, or your laptop GPU).
3. **Optional API policy** — If `USE_API_FOR_POLICY=1` and keys are set, the notebook can call an **OpenAI-compatible** endpoint for `generate` instead of local `model.generate` (saves VRAM for debugging). Default is **local** when CUDA is available.

**`MODEL_NAME` default:** `Qwen/Qwen2.5-3B-Instruct` (override for `Qwen/Qwen2.5-7B-Instruct` if you have budget).

**HF ~$30 credits (rough guide)**

| Goal | GPU | Notes for 3B |
|------|-----|----------------|
| Default | **L4** or **A10G** | bf16/fp16 + LoRA, comfortable headroom |
| Stretch | **T4** | 3B often fits bf16 LoRA; fall back to 4-bit QLoRA if OOM |
| Avoid | **A100** long jobs | High $/hr for small LoRA gains |

Stop GPU billing when idle. Router inference is only used if `USE_API_FOR_POLICY=1`.

**Static JSONL** (optional warm-start): the next cell still loads oracle-aligned rows from `india_tax_capture/data/india_tax_rows.jsonl` for offline inspection; **Phase B onward** uses **live Space rewards**.


In [7]:
# --- Colab bootstrap (no-op locally) ---
# When opened in Google Colab via the README badge, this cell clones the
# repo, installs notebook deps, and pulls secrets so the rest of the
# notebook works the same as a local checkout.

import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_OWNER = os.environ.get("INDIA_TAX_BENCH_REPO_OWNER", "Prateedk")
REPO_NAME = os.environ.get("INDIA_TAX_BENCH_REPO_NAME", "IndiaTaxBench-OpenEnv")
REPO_BRANCH = os.environ.get("INDIA_TAX_BENCH_REPO_BRANCH", "main")

if IS_COLAB:
    repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, repo_url], check=True)
    get_ipython().run_line_magic("cd", REPO_NAME)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
        check=True,
    )
    try:
        from google.colab import userdata  # type: ignore

        for k in ("HF_TOKEN", "OPENAI_API_KEY", "API_KEY", "ENV_URL", "MODEL_NAME"):
            try:
                v = userdata.get(k)
            except Exception:
                v = None
            if v:
                os.environ[k] = v
    except Exception as exc:
        print("Colab Secrets unavailable:", exc)
    print("Colab bootstrap done. cwd=", Path.cwd())
else:
    print("Local run detected (not Colab); skipping bootstrap.")

Local run detected (not Colab); skipping bootstrap.


In [8]:
import os

# DO NOT hard-code real tokens here. Pick one of the safe options:
#   • Local: `export HF_TOKEN=hf_xxx` in your shell before launching Jupyter.
#   • Colab: add HF_TOKEN under Tools → Secrets; the bootstrap cell above
#     copies it into os.environ for you.
#
# This cell is a no-op safety check that just confirms the token is set
# (does NOT print the value).

if "HF_TOKEN" not in os.environ:
    print("HF_TOKEN not set — set it via shell env or Colab Secret before running training cells.")
else:
    print("HF_TOKEN is set (value hidden).")

HF_TOKEN is set (value hidden).


In [9]:
import importlib.util
import os
import sys
from pathlib import Path

# Colab / local: pip if needed
# !pip install -q "uv"  # optional
# !pip install -q transformers peft accelerate bitsandbytes trl torch matplotlib datasets openai requests ipykernel
#
# Cursor / Jupyter: register this venv as a named kernel (once), then pick it under Select Notebook Kernel:
#   uv run python -m ipykernel install --user --name openenv-india-tax-bench --display-name "openenv-india-tax-bench (IndiaTaxBench)"

REPO_ROOT = Path.cwd() if "google.colab" in sys.modules else Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- defaults (override with environment variables) ---
os.environ.setdefault("MODEL_NAME", "Qwen/Qwen2.5-3B-Instruct")
os.environ.setdefault(
    "ENV_URL",
    "https://prateekdebit-india-tax-bench-openenv.hf.space",
)
os.environ.setdefault("HTTP_TIMEOUT", "120")
os.environ.setdefault("MAX_ROLLOUT_TASKS", "")  # empty = all tasks from server.tasks
os.environ.setdefault("USE_API_FOR_POLICY", "0")  # "1" to use OpenAI-compatible API for policy
os.environ.setdefault("API_BASE_URL", "https://router.huggingface.co/v1")

MODEL_NAME = os.environ["MODEL_NAME"]
ENV_URL = os.environ["ENV_URL"].rstrip("/")
HTTP_TIMEOUT = float(os.environ["HTTP_TIMEOUT"])
_MAXR = os.environ.get("MAX_ROLLOUT_TASKS", "").strip()
MAX_ROLLOUT_TASKS = int(_MAXR) if _MAXR.isdigit() else None
USE_API_FOR_POLICY = os.environ.get("USE_API_FOR_POLICY", "0").strip().lower() in ("1", "true", "yes")
API_BASE_URL = os.environ["API_BASE_URL"]

JSONL = REPO_ROOT / "india_tax_capture" / "data" / "india_tax_rows.jsonl"
PLOT_DIR = REPO_ROOT / "notebooks" / "training_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
ROLLOUT_JSONL = PLOT_DIR / "env_rollouts.jsonl"

# Load notebook helpers (no OpenAI import inside)
_spec = importlib.util.spec_from_file_location(
    "notebook_env_helpers",
    REPO_ROOT / "scripts" / "notebook_env_helpers.py",
)
_nbh = importlib.util.module_from_spec(_spec)
assert _spec.loader
_spec.loader.exec_module(_nbh)

print("REPO_ROOT=", REPO_ROOT)
print("MODEL_NAME=", MODEL_NAME)
print("ENV_URL=", ENV_URL)
print("USE_API_FOR_POLICY=", USE_API_FOR_POLICY)
print("PLOT_DIR=", PLOT_DIR)


REPO_ROOT= /Users/salonisisodiya/Prateek/IndiaTaxBench-OpenEnv
MODEL_NAME= Qwen/Qwen2.5-3B-Instruct
ENV_URL= https://prateekdebit-india-tax-bench-openenv.hf.space
USE_API_FOR_POLICY= False
PLOT_DIR= /Users/salonisisodiya/Prateek/IndiaTaxBench-OpenEnv/notebooks/training_plots


In [ ]:
import json
from pathlib import Path

def load_sft_rows(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        resp = obj.get("response") or {}
        if resp.get("_error"):
            continue
        old = (resp.get("tax_liability") or {}).get("old_regime") or {}
        comp = old.get("components") or {}
        scenario = (obj.get("request") or {}).get("scenario") or {}
        labels = {
            "total": old.get("total"),
            "initial_tax": comp.get("initial_tax"),
            "surcharge": comp.get("surcharge"),
            "cess": comp.get("cess"),
        }
        user = (
            "FY 2024-25 India income tax, OLD regime. Scenario JSON:\n"
            + json.dumps(scenario, ensure_ascii=False)
            + "\nRespond with JSON only: total, initial_tax, surcharge, cess (numbers)."
        )
        assistant = json.dumps(labels, ensure_ascii=False)
        rows.append({"messages": [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]})
    return rows

examples = load_sft_rows(JSONL)
len(examples), examples[0]["messages"][0]["content"][:200]


## Phase A — Space smoke test

`POST /reset` and print `task_id` / `task_difficulty` from the observation.


In [ ]:
import requests
from server.tasks import ALL_TASK_IDS

assert ALL_TASK_IDS, "No tasks in JSONL — cannot reset"
probe_task = ALL_TASK_IDS[0]
r = requests.post(
    f"{ENV_URL}/reset",
    json={"task": probe_task},
    timeout=HTTP_TIMEOUT,
)
r.raise_for_status()
payload = r.json()
obs = _nbh.unwrap_observation(payload)
print("HTTP", r.status_code, "task_id=", obs.get("task_id"), "difficulty=", obs.get("task_difficulty"))
print("valid_actions (sample)=", obs.get("valid_actions"))


## Phase B — Rollouts (local `generate` or API) + reward plot

Mirrors `inference.py` control flow: submit → optional revise → finalize. Rewards come **only** from the Space.


In [ ]:
import json
from typing import Any, Callable, Dict, List, Optional

import matplotlib.pyplot as plt
import requests

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

REVISE_THRESHOLD = 0.85


def env_reset(task_id: str) -> dict:
    resp = requests.post(
        f"{ENV_URL}/reset", json={"task": task_id}, timeout=HTTP_TIMEOUT
    )
    resp.raise_for_status()
    return resp.json()


def env_step(action: Dict[str, Any]) -> dict:
    resp = requests.post(
        f"{ENV_URL}/step", json={"action": action}, timeout=HTTP_TIMEOUT
    )
    resp.raise_for_status()
    return resp.json()


def _local_complete(model, tokenizer, device: torch.device, messages: List[Dict[str, str]], max_new_tokens: int = 256) -> str:
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    enc = tokenizer(text, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0, enc["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


_api_client = None


def _api_complete(messages: List[Dict[str, str]], max_tokens: int = 512) -> str:
    global _api_client
    from openai import OpenAI

    if _api_client is None:
        key = (
            os.getenv("OPENAI_API_KEY")
            or os.getenv("HF_TOKEN")
            or os.getenv("API_KEY")
            or ""
        ).strip()
        if not key:
            raise ValueError("Set OPENAI_API_KEY, HF_TOKEN, or API_KEY for API policy mode")
        _api_client = OpenAI(base_url=API_BASE_URL, api_key=key)
    rsp = _api_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return rsp.choices[0].message.content.strip()


def run_one_episode(
    task_id: str,
    *,
    use_api: bool,
    model=None,
    tokenizer=None,
    device: Optional[torch.device] = None,
) -> Dict[str, Any]:
    def complete(messages: List[Dict[str, str]]) -> str:
        if use_api:
            return _api_complete(messages)
        assert model and tokenizer and device is not None
        return _local_complete(model, tokenizer, device, messages)

    reset_data = env_reset(task_id)
    obs = _nbh.unwrap_observation(reset_data)
    scenario = obs.get("scenario_json", "")
    rewards: List[float] = []
    raw_initial = ""

    msgs = _nbh.build_predict_messages(scenario)
    raw_initial = complete(msgs)
    pred = _nbh.parse_prediction(raw_initial)
    action = {
        "action_type": "submit_prediction",
        "predicted_total": pred.get("total", 0.0),
        "predicted_initial_tax": pred.get("initial_tax", 0.0),
        "predicted_surcharge": pred.get("surcharge", 0.0),
        "predicted_cess": pred.get("cess", 0.0),
    }
    step_data = env_step(action)
    r, done, step_obs = _nbh.step_reward_done(step_data)
    rewards.append(r)

    if not done:
        submitted = step_obs.get("submitted_predictions", [])
        mean_score = float(submitted[-1].get("score", 0.0)) if submitted else 0.0
        if mean_score < REVISE_THRESHOLD and submitted:
            feedback = step_obs.get("feedback", "")
            prev = json.dumps(submitted[-1])
            rmsgs = _nbh.build_revise_messages(scenario, prev, feedback)
            raw_rev = complete(rmsgs)
            rp = _nbh.parse_prediction(raw_rev)
            rev_action = {
                "action_type": "revise_prediction",
                "item_index": 0,
                "predicted_total": rp.get("total", 0.0),
                "predicted_initial_tax": rp.get("initial_tax", 0.0),
                "predicted_surcharge": rp.get("surcharge", 0.0),
                "predicted_cess": rp.get("cess", 0.0),
            }
            step_data = env_step(rev_action)
            r, done, step_obs = _nbh.step_reward_done(step_data)
            rewards.append(r)

    if not done:
        step_data = env_step({"action_type": "finalize"})
        r, done, step_obs = _nbh.step_reward_done(step_data)
        rewards.append(r)

    final_reward = rewards[-1] if rewards else 0.0
    return {
        "task_id": task_id,
        "rewards": rewards,
        "final_reward": float(final_reward),
        "scenario_json": scenario[:2000],
        "raw_initial": raw_initial[:4000],
        "last_feedback": step_obs.get("feedback", "")[:2000],
    }


# --- choose policy backend ---
use_cuda = torch.cuda.is_available()
use_api = USE_API_FOR_POLICY or (not use_cuda)

model = tokenizer = None
device: Optional[torch.device] = None
if not use_api:
    device = torch.device("cuda")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map={"": 0},
    )
    lora = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"]
    )
    model = get_peft_model(model, lora)
    model.eval()
    print("Policy: LOCAL", MODEL_NAME)
else:
    print("Policy: API (USE_API_FOR_POLICY or no CUDA)", MODEL_NAME)

# --- roll all tasks (subset optional) ---
ids = list(ALL_TASK_IDS)
if MAX_ROLLOUT_TASKS is not None:
    ids = ids[: max(0, MAX_ROLLOUT_TASKS)]

rollout_rows: List[Dict[str, Any]] = []
for tid in ids:
    row = run_one_episode(
        tid,
        use_api=use_api,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    rollout_rows.append(row)
    print(tid, "final_reward=", round(row["final_reward"], 4), "rewards=", [round(x, 4) for x in row["rewards"]])

# --- plot + save ---
plt.figure(figsize=(8, 4))
plt.bar([r["task_id"] for r in rollout_rows], [r["final_reward"] for r in rollout_rows])
plt.xticks(rotation=45, ha="right")
plt.ylabel("finalize / last step reward")
plt.title("IndiaTaxBench Space rollouts (before SFT)")
plt.tight_layout()
fig_before = PLOT_DIR / "rewards_before_sft.png"
plt.savefig(fig_before, dpi=150)
plt.show()
print("Saved", fig_before)


## Phase C — `env_rollouts.jsonl` for SFT

Build chat-style rows from rollout trajectories (initial model output as assistant target).


In [ ]:
def rollout_to_sft_messages(row: Dict[str, Any]) -> Dict[str, Any]:
    scenario = row.get("scenario_json", "")
    user = (
        "FY 2024-25 India income tax, OLD regime. Scenario JSON:\n"
        + scenario
        + "\nRespond with JSON only: total, initial_tax, surcharge, cess (numbers)."
    )
    assistant = row.get("raw_initial", "").strip()
    return {"messages": [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]}

sft_from_env = [rollout_to_sft_messages(r) for r in rollout_rows]
ROLLOUT_JSONL.write_text(
    "\n".join(json.dumps(x, ensure_ascii=False) for x in sft_from_env) + "\n",
    encoding="utf-8",
)
print("Wrote", ROLLOUT_JSONL, "n=", len(sft_from_env))


### Optional: Unsloth (faster LoRA)

If you use a Colab/HF GPU image with [Unsloth](https://github.com/unslothai/unsloth) preinstalled, you can swap the TRL block for Unsloth's fast LoRA path. This repo stub uses **TRL only** for a single `pip install` story (`uv sync --extra notebook`).


## Phase D — TRL `SFTTrainer` stub (GPU)

Tiny `max_steps` for smoke; increase on a paid GPU session. **Before/after** compares mean final reward on two tasks using the same rollout driver.


In [ ]:
mean_before = sum(r["final_reward"] for r in rollout_rows) / max(1, len(rollout_rows))
print("mean final reward (before SFT)=", round(mean_before, 4))

trained_model = model
if model is not None and use_cuda and not use_api:
    try:
        from datasets import Dataset
        from trl import SFTConfig, SFTTrainer

        train_ds = Dataset.from_list(sft_from_env)

        def formatting_func(example):
            return tokenizer.apply_chat_template(
                example["messages"], tokenize=False, add_generation_prompt=False
            )

        sft_config = SFTConfig(
            output_dir=str(PLOT_DIR / "sft_out"),
            max_steps=20,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,
            learning_rate=2e-4,
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=5,
            report_to=[],
            max_seq_length=2048,
        )

        trainer = SFTTrainer(
            model=model,
            args=sft_config,
            train_dataset=train_ds,
            processing_class=tokenizer,
            formatting_func=formatting_func,
        )
        trainer.train()
        trained_model = trainer.model
        print("SFT stub finished.")
    except Exception as exc:
        print("SFTTrainer skipped:", exc)
else:
    print("SFT stub skipped (API policy or no CUDA).")

# --- after: re-roll subset ---
compare_ids = ids[: min(2, len(ids))]
after_rows = []
eval_model = trained_model if trained_model is not None else model
for tid in compare_ids:
    row = run_one_episode(
        tid,
        use_api=use_api,
        model=eval_model,
        tokenizer=tokenizer,
        device=device,
    )
    after_rows.append(row)
    print("after", tid, "final_reward=", round(row["final_reward"], 4))

mean_after = sum(r["final_reward"] for r in after_rows) / max(1, len(after_rows))
print("mean final reward (after, subset)=", round(mean_after, 4))

labels = ["before\n(full set)", "after\n(subset)"]
vals = [mean_before, mean_after]
plt.figure(figsize=(5, 4))
plt.bar(labels, vals)
plt.ylabel("mean final reward")
plt.title("Smoke before/after SFT (subset may differ)")
plt.tight_layout()
fig_cmp = PLOT_DIR / "reward_before_after_mean.png"
plt.savefig(fig_cmp, dpi=150)
plt.show()
print("Saved", fig_cmp)
